In [3]:
import sys
from pathlib import Path
import pandas as pd
root=Path.cwd().parent
print(root)

c:\Users\USER\Desktop\data-analysis\Data-analysis


In [5]:
sys.path.append(str(root))
from src.ingestion.csv_ingestion import load_datasets
from src.validation.data_quality import check_missing_values, check_duplicates, check_unique_values, check_data_type, check_value_range, check_categorical_values, check_date_order, check_outliers, check_referential_integrity

In [6]:
datasets=load_datasets()
results = []

In [7]:
for name, df in datasets.items():
    results.append(check_missing_values(df, name))
    results.append(check_duplicates(df, name))

In [8]:
keys = [
    ("olist_customers_dataset", ["customer_id"]),
    ("olist_orders_dataset", ["order_id"]),
    ("olist_products_dataset", ["product_id"]),
    ("olist_sellers_dataset", ["seller_id"]),
    ("olist_order_items_dataset", ["order_id", "order_item_id"]),
    ("olist_order_reviews_dataset", ["order_id"]),
    ("olist_order_reviews_dataset", ["review_id"]),
    ("product_category_name_translation", ["product_category_name"]),
]
for name, key in keys:
    results.append(check_unique_values(datasets[name], name, key))

In [9]:
types = [
    ("olist_customers_dataset", "customer_zip_code_prefix", str),
    ("olist_sellers_dataset", "seller_zip_code_prefix", str),
    ("olist_geolocation_dataset", "geolocation_zip_code_prefix", str),
    ("olist_order_items_dataset", "shipping_limit_date", "datetime64[ns]"),
    ("olist_order_reviews_dataset", "review_creation_date", "datetime64[ns]"),
    ("olist_order_reviews_dataset", "review_answer_timestamp", "datetime64[ns]"),
    ("olist_orders_dataset", "order_purchase_timestamp", "datetime64[ns]"),
    ("olist_orders_dataset", "order_approved_at", "datetime64[ns]"),
    ("olist_orders_dataset", "order_delivered_carrier_date", "datetime64[ns]"),
    ("olist_orders_dataset", "order_delivered_customer_date", "datetime64[ns]"),
    ("olist_orders_dataset", "order_estimated_delivery_date", "datetime64[ns]"),
]
for name, column, dtype in types:
    results.append(check_data_type(datasets[name], name, column, dtype))


In [10]:
ranges = [
    ("olist_order_payments_dataset", "payment_installments", 1, None),
    ("olist_order_payments_dataset", "payment_value", 0.01, None),
    ("olist_order_items_dataset", "price", 0.01, None),
    ("olist_order_items_dataset", "freight_value", 0.01, None),
    ("olist_products_dataset", "product_weight_g", 1, None),
    ("olist_order_reviews_dataset", "review_score", 1, 5),
    ("olist_geolocation_dataset", "geolocation_lat", -33.8, 5.3),
    ("olist_geolocation_dataset", "geolocation_lng", -73.9, -34.8),
]
for name, column, low, high in ranges:
    results.append(check_value_range(datasets[name], name, column, low, high))


In [11]:
categoricals = [
    ("olist_orders_dataset", "order_status"),
    ("olist_order_payments_dataset", "payment_type"),
    ("olist_customers_dataset", "customer_state"),
    ("olist_customers_dataset", "customer_city"),
    ("olist_sellers_dataset", "seller_state"),
    ("olist_sellers_dataset", "seller_city"),
    ("olist_geolocation_dataset", "geolocation_state"),
    ("olist_geolocation_dataset", "geolocation_city"),
    ("olist_products_dataset", "product_category_name"),
]
for name, column in categoricals:
    results.append(check_categorical_values(datasets[name], name, column))

In [12]:
date_pairs = [
    ("olist_orders_dataset", "order_purchase_timestamp", "order_approved_at"),
    ("olist_orders_dataset", "order_approved_at", "order_delivered_carrier_date"),
    ("olist_orders_dataset", "order_delivered_carrier_date", "order_delivered_customer_date"),
    ("olist_orders_dataset", "order_purchase_timestamp", "order_delivered_customer_date"),
    ("olist_order_reviews_dataset", "review_creation_date", "review_answer_timestamp"),
]
for name, earlier, later in date_pairs:
    results.append(check_date_order(datasets[name], name, earlier, later))

In [13]:
outlier_columns = [
    ("olist_order_items_dataset", "price"),
    ("olist_order_items_dataset", "freight_value"),
    ("olist_order_payments_dataset", "payment_value"),
    ("olist_products_dataset", "product_weight_g"),
]
for name, column in outlier_columns:
    results.append(check_outliers(datasets[name], name, column))

In [14]:
links = [
    ("olist_orders_dataset", "olist_customers_dataset", "customer_id", "customer_id"),
    ("olist_order_items_dataset", "olist_orders_dataset", "order_id", "order_id"),
    ("olist_order_items_dataset", "olist_products_dataset", "product_id", "product_id"),
    ("olist_order_items_dataset", "olist_sellers_dataset", "seller_id", "seller_id"),
    ("olist_order_payments_dataset", "olist_orders_dataset", "order_id", "order_id"),
    ("olist_order_reviews_dataset", "olist_orders_dataset", "order_id", "order_id"),
    ("olist_products_dataset", "product_category_name_translation",
     "product_category_name", "product_category_name"),
    ("olist_orders_dataset", "olist_order_items_dataset", "order_id", "order_id"),
    ("olist_orders_dataset", "olist_order_payments_dataset", "order_id", "order_id"),
    ("olist_orders_dataset", "olist_order_reviews_dataset", "order_id", "order_id"),
    ("olist_customers_dataset", "olist_geolocation_dataset",
     "customer_zip_code_prefix", "geolocation_zip_code_prefix"),
    ("olist_sellers_dataset", "olist_geolocation_dataset",
     "seller_zip_code_prefix", "geolocation_zip_code_prefix"),
]
for child, parent, child_col, parent_col in links:
    results.append(check_referential_integrity(
        datasets[child], datasets[parent], child, parent, child_col, parent_col))

report = pd.concat(results, ignore_index=True)
print(report.shape)
print(report.to_string())

(47, 7)
                           table                  check                                                       column   count percent severity                                                                                                                          message
0      olist_geolocation_dataset             duplicates                                                          all  261831   26.18  warning                                                                                                   261831 duplicate rows (26.18%)
1           olist_orders_dataset         missing_values                                            order_approved_at     160    0.16  warning                                                                                    order_approved_at: 160 missing values (0.16%)
2           olist_orders_dataset         missing_values                                 order_delivered_carrier_date    1783    1.79  warning                                      